In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# !pip install optuna

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, DistilBertForSequenceClassification
from sklearn.metrics import mean_squared_error, mean_absolute_error, cohen_kappa_score
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from transformers import (AutoConfig,
                          AutoModelForSequenceClassification,
                          AutoTokenizer, AdamW,
                          get_linear_schedule_with_warmup,
                          set_seed,
                          )
# import optuna
# from optuna.integration import PyTorchLightningPruningCallback

In [4]:

import torch
import pandas as pd
from torch.utils.data import DataLoader, Dataset
from transformers import RobertaTokenizer, RobertaForSequenceClassification, AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, cohen_kappa_score, r2_score
from torch.optim.lr_scheduler import StepLR
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [5]:
!pip install indic-nlp-library

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 67.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 18.1 MB/s eta 0:00:00


In [146]:
df = pd.read_csv("/content/drive/MyDrive/EG/translated_dataset/google_translator/nep_p8.csv")

In [147]:
df.head(2)

,Unnamed: 0.1,Unnamed: 0,essay_id,essay_set,essay,score,essay_hindi,p8_google_trans
0,0,12255,20716,8,A long time ago when I was in third grade I h...,34,बहुत समय पहले जब मैं तीसरी कक्षा में था तो मेर...,धेरै पहिले जब म तेस्रो कक्षामा थिएँ मेरो एक सा...
1,1,12256,20717,8,Softball has to be one of the single most gre...,46,सॉफ्टबॉल को जीवित सबसे महान खेलों में से एक हो...,सफ्टबल जीवित सबैभन्दा ठूलो खेल मध्ये एक हुनुपर...


In [148]:
import re
Tweet = []
for tweettext in df["p8_google_trans"]:
  text = re.sub(r"http\S+", "", tweettext)
  text = re.sub(r"@\S+","",text)
  emoji_pattern = re.compile("["
                               u"\U0001F600-\U0001F64F"  # emoticons
                               u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                               u"\U0001F680-\U0001F6FF"  # transport & map symbols
                               u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                               u"\U00002500-\U00002BEF"  # chinese char
                               u"\U00002702-\U000027B0"
                               u"\U00002702-\U000027B0"
                               u"\U000024C2-\U0001F251"
                               u"\U0001f926-\U0001f937"
                               u"\U00010000-\U0010ffff"
                               u"\u2640-\u2642"
                               u"\u2600-\u2B55"
                               u"\u200d"
                               u"\u23cf"
                               u"\u23e9"
                               u"\u231a"
                               u"\ufe0f"  # dingbats
                               u"\u3030"
                               "]+", flags=re.UNICODE)

  text = emoji_pattern.sub(r'',text)
  text = re.sub('[A-Za-z]+', ' ', text) #Remove english alphabets
  #print(text)
  Tweet.append(text)

In [149]:
df["p8_google_trans"] = pd.Series(Tweet)
df.head()

,Unnamed: 0.1,Unnamed: 0,essay_id,essay_set,essay,score,essay_hindi,p8_google_trans
0,0,12255,20716,8,A long time ago when I was in third grade I h...,34,बहुत समय पहले जब मैं तीसरी कक्षा में था तो मेर...,धेरै पहिले जब म तेस्रो कक्षामा थिएँ मेरो एक सा...
1,1,12256,20717,8,Softball has to be one of the single most gre...,46,सॉफ्टबॉल को जीवित सबसे महान खेलों में से एक हो...,सफ्टबल जीवित सबैभन्दा ठूलो खेल मध्ये एक हुनुपर...
2,2,12257,20718,8,"Some people like making people laugh, I love ...",40,"कुछ लोग लोगों को हंसाना पसंद करते हैं, मुझे यह...","केही मानिसहरू मानिसहरूलाई हँसाउन मन पराउँछन्, ..."
3,3,12258,20719,8,"""LAUGHTER"" @CAPS1 I hang out with my friends...",30,"""हँसी"" @ CAPS1 मैं अपने दोस्तों के साथ घूमता ह...","""हाँसो"" म मेरा साथीहरूसँग ह्याङ्ग आउट गर्छु, ..."
4,4,12259,20721,8,Well ima tell a story about the time i got @CA...,26,खैर इमा उस समय के बारे में एक कहानी बताएं जब म...,"ठिकै छ, मैले डाउन टाउन पाएको समयको बारेमा एउट..."


In [150]:
df['p8_google_trans'] = df['p8_google_trans'].replace(r'\r+|\n+|\t+','', regex=True)
df['p8_google_trans'] = df['p8_google_trans'].str.replace("  ", " ")

In [151]:


# tokenization
from indicnlp.tokenize import indic_tokenize
def tokenization(indic_string):
    tokens = []
    for t in indic_tokenize.trivial_tokenize(indic_string):
        tokens.append(t)
    return tokens
df['p8_google_trans'] = df['p8_google_trans'].apply(lambda x: tokenization(x))

In [152]:
df.head(5)

,Unnamed: 0.1,Unnamed: 0,essay_id,essay_set,essay,score,essay_hindi,p8_google_trans
0,0,12255,20716,8,A long time ago when I was in third grade I h...,34,बहुत समय पहले जब मैं तीसरी कक्षा में था तो मेर...,"[धेरै, पहिले, जब, म, तेस्रो, कक्षामा, थिएँ, मे..."
1,1,12256,20717,8,Softball has to be one of the single most gre...,46,सॉफ्टबॉल को जीवित सबसे महान खेलों में से एक हो...,"[सफ्टबल, जीवित, सबैभन्दा, ठूलो, खेल, मध्ये, एक..."
2,2,12257,20718,8,"Some people like making people laugh, I love ...",40,"कुछ लोग लोगों को हंसाना पसंद करते हैं, मुझे यह...","[केही, मानिसहरू, मानिसहरूलाई, हँसाउन, मन, पराउ..."
3,3,12258,20719,8,"""LAUGHTER"" @CAPS1 I hang out with my friends...",30,"""हँसी"" @ CAPS1 मैं अपने दोस्तों के साथ घूमता ह...","["", हाँसो, "", म, मेरा, साथीहरूसँग, ह्याङ्ग, आउ..."
4,4,12259,20721,8,Well ima tell a story about the time i got @CA...,26,खैर इमा उस समय के बारे में एक कहानी बताएं जब म...,"[ठिकै, छ, ,, मैले, डाउन, टाउन, पाएको, समयको, ब..."


In [153]:
# Remove ‘\n’ from each tokenized
for i in range(len(df)):
    df['p8_google_trans'][i] = [s.replace("\n", "") for s in df['p8_google_trans'][i]]

<ipython-input-153-e8b962c7e802>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['p8_google_trans'][i] = [s.replace("\n", "") for s in df['p8_google_trans'][i]]


In [154]:
# Remove Punctuations

punctuations = ['nn','n','–', '।','/', '`', '+', '\\', '"', '?', '▁(', '$', '@', '[', '_', "\'", '!', ',', ':', '^', '|', ']', '=', '%', '&', '.', ')', '(', "#", '*', '', ';', '-', '}','|','"']


to_be_removed = punctuations

for i in range(len(df)):
    df['p8_google_trans'][i]=[ele for ele in df['p8_google_trans'][i] if ele not in (to_be_removed)]
# count_length()
df.head(2)

<ipython-input-154-b256a185b682>:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['p8_google_trans'][i]=[ele for ele in df['p8_google_trans'][i] if ele not in (to_be_removed)]


,Unnamed: 0.1,Unnamed: 0,essay_id,essay_set,essay,score,essay_hindi,p8_google_trans
0,0,12255,20716,8,A long time ago when I was in third grade I h...,34,बहुत समय पहले जब मैं तीसरी कक्षा में था तो मेर...,"[धेरै, पहिले, जब, म, तेस्रो, कक्षामा, थिएँ, मे..."
1,1,12256,20717,8,Softball has to be one of the single most gre...,46,सॉफ्टबॉल को जीवित सबसे महान खेलों में से एक हो...,"[सफ्टबल, जीवित, सबैभन्दा, ठूलो, खेल, मध्ये, एक..."


In [155]:
# #Remove comma as a seperators
for i in range(len(df)):
    df['p8_google_trans'][i] = ' '.join(df['p8_google_trans'][i])
df.head(5)

<ipython-input-155-a8f9be9f241d>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['p8_google_trans'][i] = ' '.join(df['p8_google_trans'][i])


,Unnamed: 0.1,Unnamed: 0,essay_id,essay_set,essay,score,essay_hindi,p8_google_trans
0,0,12255,20716,8,A long time ago when I was in third grade I h...,34,बहुत समय पहले जब मैं तीसरी कक्षा में था तो मेर...,धेरै पहिले जब म तेस्रो कक्षामा थिएँ मेरो एक सा...
1,1,12256,20717,8,Softball has to be one of the single most gre...,46,सॉफ्टबॉल को जीवित सबसे महान खेलों में से एक हो...,सफ्टबल जीवित सबैभन्दा ठूलो खेल मध्ये एक हुनुपर...
2,2,12257,20718,8,"Some people like making people laugh, I love ...",40,"कुछ लोग लोगों को हंसाना पसंद करते हैं, मुझे यह...",केही मानिसहरू मानिसहरूलाई हँसाउन मन पराउँछन् म...
3,3,12258,20719,8,"""LAUGHTER"" @CAPS1 I hang out with my friends...",30,"""हँसी"" @ CAPS1 मैं अपने दोस्तों के साथ घूमता ह...",हाँसो म मेरा साथीहरूसँग ह्याङ्ग आउट गर्छु हामी...
4,4,12259,20721,8,Well ima tell a story about the time i got @CA...,26,खैर इमा उस समय के बारे में एक कहानी बताएं जब म...,ठिकै छ मैले डाउन टाउन पाएको समयको बारेमा एउटा ...


In [156]:
len(df)

723

In [157]:
df = df.dropna(subset=['score'])
df.reset_index(drop=True, inplace=True)

In [158]:
print(df.isnull().any())

Unnamed: 0.1       False
Unnamed: 0         False
essay_id           False
essay_set          False
essay              False
score              False
essay_hindi        False
p8_google_trans    False
dtype: bool


In [159]:
print(df.isnull().sum().sum())

0


In [160]:
from sklearn.preprocessing import MinMaxScaler
original_values = df['score'].values.reshape(-1, 1)
scaler = MinMaxScaler(feature_range=(0, 1), copy=True)
normalized_values = scaler.fit_transform(original_values)
df['normalized_score'] = normalized_values
original_values_restored = scaler.inverse_transform(normalized_values)
df['restored_score'] = original_values_restored

In [161]:
df.head()

,Unnamed: 0.1,Unnamed: 0,essay_id,essay_set,essay,score,essay_hindi,p8_google_trans,normalized_score,restored_score
0,0,12255,20716,8,A long time ago when I was in third grade I h...,34,बहुत समय पहले जब मैं तीसरी कक्षा में था तो मेर...,धेरै पहिले जब म तेस्रो कक्षामा थिएँ मेरो एक सा...,0.48,34.0
1,1,12256,20717,8,Softball has to be one of the single most gre...,46,सॉफ्टबॉल को जीवित सबसे महान खेलों में से एक हो...,सफ्टबल जीवित सबैभन्दा ठूलो खेल मध्ये एक हुनुपर...,0.72,46.0
2,2,12257,20718,8,"Some people like making people laugh, I love ...",40,"कुछ लोग लोगों को हंसाना पसंद करते हैं, मुझे यह...",केही मानिसहरू मानिसहरूलाई हँसाउन मन पराउँछन् म...,0.60,40.0
3,3,12258,20719,8,"""LAUGHTER"" @CAPS1 I hang out with my friends...",30,"""हँसी"" @ CAPS1 मैं अपने दोस्तों के साथ घूमता ह...",हाँसो म मेरा साथीहरूसँग ह्याङ्ग आउट गर्छु हामी...,0.40,30.0
4,4,12259,20721,8,Well ima tell a story about the time i got @CA...,26,खैर इमा उस समय के बारे में एक कहानी बताएं जब म...,ठिकै छ मैले डाउन टाउन पाएको समयको बारेमा एउटा ...,0.32,26.0


In [162]:
df.p8_google_trans

0      धेरै पहिले जब म तेस्रो कक्षामा थिएँ मेरो एक सा...
1      सफ्टबल जीवित सबैभन्दा ठूलो खेल मध्ये एक हुनुपर...
2      केही मानिसहरू मानिसहरूलाई हँसाउन मन पराउँछन् म...
3      हाँसो म मेरा साथीहरूसँग ह्याङ्ग आउट गर्छु हामी...
4      ठिकै छ मैले डाउन टाउन पाएको समयको बारेमा एउटा ...
                             ...                        
718    धेरैजसो कथाहरूमा आमा र छोरीहरू या त शत्रु वा स...
719    मैले कहिल्यै बुझिन कि हाँसो दुई व्यक्तिहरू बीच...
720    जब तपाईं हाँस्नुहुन्छ बानी छैन वा कारण हो हाँस...
721    फेन्समा ट्रिपिन म वर्ष जवान हुँ र ती छोटो वर्ष...
722    धेरै मानिसहरू विश्वास गर्छन् कि हाँसोले तपाईंक...
Name: p8_google_trans, Length: 723, dtype: object

In [163]:
highest_score = df['score'].max()
print(highest_score)

60


In [164]:

highest_score = df['normalized_score'].max()
print(highest_score)

1.0


In [165]:
df['score'].value_counts()

40    161
36     65
30     49
35     47
34     39
37     39
32     37
31     34
33     32
45     31
42     23
41     22
38     20
43     15
44     14
46     13
50     13
28     11
39      8
29      8
47      7
27      6
25      5
26      4
20      4
24      3
48      3
49      2
55      2
21      2
60      1
15      1
22      1
10      1
Name: score, dtype: int64

In [166]:
df['normalized_score'].value_counts()

0.60    161
0.52     65
0.40     49
0.50     47
0.48     39
0.54     39
0.44     37
0.42     34
0.46     32
0.70     31
0.64     23
0.62     22
0.56     20
0.66     15
0.68     14
0.72     13
0.80     13
0.36     11
0.58      8
0.38      8
0.74      7
0.34      6
0.30      5
0.32      4
0.20      4
0.28      3
0.76      3
0.78      2
0.90      2
0.22      2
1.00      1
0.10      1
0.24      1
0.00      1
Name: normalized_score, dtype: int64

In [167]:
df.head()

,Unnamed: 0.1,Unnamed: 0,essay_id,essay_set,essay,score,essay_hindi,p8_google_trans,normalized_score,restored_score
0,0,12255,20716,8,A long time ago when I was in third grade I h...,34,बहुत समय पहले जब मैं तीसरी कक्षा में था तो मेर...,धेरै पहिले जब म तेस्रो कक्षामा थिएँ मेरो एक सा...,0.48,34.0
1,1,12256,20717,8,Softball has to be one of the single most gre...,46,सॉफ्टबॉल को जीवित सबसे महान खेलों में से एक हो...,सफ्टबल जीवित सबैभन्दा ठूलो खेल मध्ये एक हुनुपर...,0.72,46.0
2,2,12257,20718,8,"Some people like making people laugh, I love ...",40,"कुछ लोग लोगों को हंसाना पसंद करते हैं, मुझे यह...",केही मानिसहरू मानिसहरूलाई हँसाउन मन पराउँछन् म...,0.60,40.0
3,3,12258,20719,8,"""LAUGHTER"" @CAPS1 I hang out with my friends...",30,"""हँसी"" @ CAPS1 मैं अपने दोस्तों के साथ घूमता ह...",हाँसो म मेरा साथीहरूसँग ह्याङ्ग आउट गर्छु हामी...,0.40,30.0
4,4,12259,20721,8,Well ima tell a story about the time i got @CA...,26,खैर इमा उस समय के बारे में एक कहानी बताएं जब म...,ठिकै छ मैले डाउन टाउन पाएको समयको बारेमा एउटा ...,0.32,26.0


### amitness/roberta-base-ne

In [173]:
# Custom Dataset
class EssayDataset(Dataset):
    def __init__(self, essays, scores, tokenizer, max_len):
        self.essays = essays
        self.scores = scores
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.essays)

    def __getitem__(self, item):
        essay = str(self.essays[item])
        score = self.scores[item]

        encoding = self.tokenizer.encode_plus(
            essay,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'score': torch.tensor(score, dtype=torch.float)
        }

In [174]:
# Model and Tokenizer
tokenizer = RobertaTokenizer.from_pretrained('amitness/roberta-base-ne')
model = RobertaForSequenceClassification.from_pretrained('amitness/roberta-base-ne', num_labels=1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at amitness/roberta-base-ne and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [175]:
# Splitting data
train_texts, temp_texts, train_scores, temp_scores = train_test_split(df['p8_google_trans'], df['normalized_score'], test_size=0.3)

# Reset index
train_texts = train_texts.reset_index(drop=True)
temp_texts = temp_texts.reset_index(drop=True)
train_scores = train_scores.reset_index(drop=True)
temp_scores = temp_scores.reset_index(drop=True)

# val data and test data
val_texts, test_texts, val_scores, test_scores = train_test_split(temp_texts, temp_scores, test_size=0.5)

# Reset index
val_texts = val_texts.reset_index(drop=True)
test_texts = test_texts.reset_index(drop=True)
val_scores = val_scores.reset_index(drop=True)
test_scores = test_scores.reset_index(drop=True)


train_dataset = EssayDataset(train_texts, train_scores, tokenizer, max_len=512)
val_dataset = EssayDataset(val_texts, val_scores, tokenizer, max_len=512)
test_dataset = EssayDataset(test_texts, test_scores, tokenizer, max_len=512)

train_loader = DataLoader(train_dataset, batch_size=8)
val_loader = DataLoader(val_dataset, batch_size=8)
test_loader = DataLoader(test_dataset, batch_size=8)

In [176]:
# Training
optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        scores = batch['score'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.squeeze()
        loss = torch.nn.functional.mse_loss(logits, scores)
        total_train_loss += loss.item()
        loss.backward()
        optimizer.step()
    avg_train_loss = total_train_loss / len(train_loader)

    # Validation
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            scores = batch['score'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits.squeeze()
            loss = torch.nn.functional.mse_loss(logits, scores)
            total_val_loss += loss.item()
    avg_val_loss = total_val_loss / len(val_loader)

    print(f'Epoch {epoch + 1}/{num_epochs} | Train Loss: {avg_train_loss} | Val Loss: {avg_val_loss}')

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1/10 | Train Loss: 0.04512181304016849 | Val Loss: 0.014435429874408458
Epoch 2/10 | Train Loss: 0.03208740862464765 | Val Loss: 0.015788227785378695
Epoch 3/10 | Train Loss: 0.029930355027318 | Val Loss: 0.02218400876569961
Epoch 4/10 | Train Loss: 0.03568659019947518 | Val Loss: 0.02274993447852986
Epoch 5/10 | Train Loss: 0.02217123962327605 | Val Loss: 0.0219267142404403
Epoch 6/10 | Train Loss: 0.022729186945070978 | Val Loss: 0.012127811455034785
Epoch 7/10 | Train Loss: 0.018730145500740036 | Val Loss: 0.016402095090597868
Epoch 8/10 | Train Loss: 0.019074308904237114 | Val Loss: 0.012040296236851386
Epoch 9/10 | Train Loss: 0.01888451582453854 | Val Loss: 0.013530569284089975
Epoch 10/10 | Train Loss: 0.022211354196770117 | Val Loss: 0.012652764030333077


In [177]:
# Evaluation
def evaluate_model(model, test_loader):
    model.eval()
    predictions, actuals = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            scores = batch['score'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            predictions.extend(outputs.logits.squeeze().cpu().numpy())
            actuals.extend(scores.cpu().numpy())

    # Convert to arrays
    predictions = np.array(predictions)
    actuals = np.array(actuals)

    predictions_inv_tras = (scaler.inverse_transform(predictions.reshape(-1, 1)).squeeze())
    actuals_inv_tras = (scaler.inverse_transform(actuals.reshape(-1, 1)).squeeze())

    mse = mean_squared_error(actuals, predictions)
    mae = mean_absolute_error(actuals, predictions)
    r2 = r2_score(actuals, predictions)
    pearson_corr = pearsonr(actuals, predictions)[0]
    spearman_corr = spearmanr(actuals, predictions)[0]
    qwk = cohen_kappa_score(np.round(actuals_inv_tras), np.round(predictions_inv_tras), weights='quadratic')

    return mse, mae, r2, pearson_corr, spearman_corr, qwk

mse, mae, r2, pearson_corr, spearman_corr, qwk = evaluate_model(model, test_loader)
print(f'MSE: {mse}')
print(f'MAE: {mae}')
print(f'R2: {r2}')
print(f'Pearson Correlation: {pearson_corr}')
print(f'Spearman Correlation: {spearman_corr}')
print(f'QWK: {qwk}')

MSE: 0.013381442055106163
MAE: 0.09044664353132248
R2: -0.14170649499053312
Pearson Correlation: 0.3438850352126217
Spearman Correlation: 0.35507697170382196
QWK: 0.2820028559583583




MSE: 0.01683727838099003
MAE: 0.09970073401927948
R2: 0.36088829195233807
Pearson Correlation: 0.6541778233775262
Spearman Correlation: 0.5376987551112392
QWK: 0.6119408735407811

### NepBERTa/NepBERTa

In [135]:
# Custom Dataset
class EssayDataset(Dataset):
    def __init__(self, essays, scores, tokenizer, max_len):
        self.essays = essays
        self.scores = scores
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.essays)

    def __getitem__(self, item):
        essay = str(self.essays[item])
        score = self.scores[item]

        encoding = self.tokenizer.encode_plus(
            essay,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'score': torch.tensor(score, dtype=torch.float)
        }


tokenizer = BertTokenizer.from_pretrained('NepBERTa/NepBERTa')
model = BertForSequenceClassification.from_pretrained('NepBERTa/NepBERTa', num_labels=1, from_tf=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Splitting data
train_texts, temp_texts, train_scores, temp_scores = train_test_split(df['p8_google_trans'], df['normalized_score'], test_size=0.3)

# Reset index
train_texts = train_texts.reset_index(drop=True)
temp_texts = temp_texts.reset_index(drop=True)
train_scores = train_scores.reset_index(drop=True)
temp_scores = temp_scores.reset_index(drop=True)

# val data and test data
val_texts, test_texts, val_scores, test_scores = train_test_split(temp_texts, temp_scores, test_size=0.5)

# Reset index
val_texts = val_texts.reset_index(drop=True)
test_texts = test_texts.reset_index(drop=True)
val_scores = val_scores.reset_index(drop=True)
test_scores = test_scores.reset_index(drop=True)


train_dataset = EssayDataset(train_texts, train_scores, tokenizer, max_len=512)
val_dataset = EssayDataset(val_texts, val_scores, tokenizer, max_len=512)
test_dataset = EssayDataset(test_texts, test_scores, tokenizer, max_len=512)

train_loader = DataLoader(train_dataset, batch_size=8)
val_loader = DataLoader(val_dataset, batch_size=8)
test_loader = DataLoader(test_dataset, batch_size=8)

# Training
optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        scores = batch['score'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.squeeze()
        loss = torch.nn.functional.mse_loss(logits, scores)
        total_train_loss += loss.item()
        loss.backward()
        optimizer.step()
    avg_train_loss = total_train_loss / len(train_loader)

    # Validation
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            scores = batch['score'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits.squeeze()
            loss = torch.nn.functional.mse_loss(logits, scores)
            total_val_loss += loss.item()
    avg_val_loss = total_val_loss / len(val_loader)

    print(f'Epoch {epoch + 1}/{num_epochs} | Train Loss: {avg_train_loss} | Val Loss: {avg_val_loss}')

# max_score = df['score'].max()
# Evaluation
def evaluate_model(model, test_loader):
    model.eval()
    predictions, actuals = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            scores = batch['score'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            predictions.extend(outputs.logits.squeeze().cpu().numpy())
            actuals.extend(scores.cpu().numpy())

    predictions = np.array(predictions)
    actuals = np.array(actuals)

    # print("Predictions: ", predictions)
    # print("Actuals: ", actuals)

    predictions_inv_tras = (scaler.inverse_transform(predictions.reshape(-1, 1)).squeeze())
    actuals_inv_tras = (scaler.inverse_transform(actuals.reshape(-1, 1)).squeeze())
    # print("Predictions_inv_tras: ", predictions_inv_tras)
    # print("Actuals_inv_tras: ", actuals_inv_tras)


    mse = mean_squared_error(actuals, predictions)
    mae = mean_absolute_error(actuals, predictions)
    r2 = r2_score(actuals, predictions)
    pearson_corr = pearsonr(actuals, predictions)[0]
    spearman_corr = spearmanr(actuals, predictions)[0]
    qwk = cohen_kappa_score(np.round(actuals_inv_tras), np.round(predictions_inv_tras), weights='quadratic')

    return mse, mae, r2, pearson_corr, spearman_corr, qwk

mse, mae, r2, pearson_corr, spearman_corr, qwk = evaluate_model(model, test_loader)
print(f'MSE: {mse}')
print(f'MAE: {mae}')
print(f'R2: {r2}')
print(f'Pearson Correlation: {pearson_corr}')
print(f'Spearman Correlation: {spearman_corr}')
print(f'QWK: {qwk}')

All TF 2.0 model weights were used when initializing BertForSequenceClassification.

All the weights of BertForSequenceClassification were initialized from the TF 2.0 model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use BertForSequenceClassification for predictions without further training.
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1/10 | Train Loss: 0.024081393206870416 | Val Loss: 0.011903854319825768
Epoch 2/10 | Train Loss: 0.0164687176948064 | Val Loss: 0.009475624521395989
Epoch 3/10 | Train Loss: 0.010730461905041011 | Val Loss: 0.010332856460341386
Epoch 4/10 | Train Loss: 0.01010960682106088 | Val Loss: 0.008292737766169012
Epoch 5/10 | Train Loss: 0.009078710347239394 | Val Loss: 0.007826054219289549
Epoch 6/10 | Train Loss: 0.008354684025107417 | Val Loss: 0.007924208716888512
Epoch 7/10 | Train Loss: 0.007360138684816775 | Val Loss: 0.007174973209787693
Epoch 8/10 | Train Loss: 0.006742570165442885 | Val Loss: 0.015105999939675843
Epoch 9/10 | Train Loss: 0.005707976808025705 | Val Loss: 0.006613418692722917
Epoch 10/10 | Train Loss: 0.007845880016247975 | Val Loss: 0.008580937482682722
MSE: 0.008312653750181198
MAE: 0.06940390914678574
R2: 0.07981236575022599
Pearson Correlation: 0.6024866050513107
Spearman Correlation: 0.5976906696070751
QWK: 0.5552821857586334


MSE: 0.014094462618231773
MAE: 0.0913693904876709
R2: 0.4616231489504393
Pearson Correlation: 0.8312037298894959
Spearman Correlation: 0.7699140596696659
QWK: 0.6625091953401965

In [136]:
print("Train dataset: ", len(train_dataset))
print("Val dataset: ", len(val_dataset))
print("Val dataset: ", len(test_dataset))

Train dataset:  506
Val dataset:  108
Val dataset:  109


### Shushant/NepNewsBERT

In [137]:
# Custom Dataset
class EssayDataset(Dataset):
    def __init__(self, essays, scores, tokenizer, max_len):
        self.essays = essays
        self.scores = scores
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.essays)

    def __getitem__(self, item):
        essay = str(self.essays[item])
        score = self.scores[item]

        encoding = self.tokenizer.encode_plus(
            essay,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'score': torch.tensor(score, dtype=torch.float)
        }

tokenizer = BertTokenizer.from_pretrained('Shushant/NepNewsBERT')
model = BertForSequenceClassification.from_pretrained('Shushant/NepNewsBERT', num_labels=1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Splitting data
train_texts, temp_texts, train_scores, temp_scores = train_test_split(df['p8_google_trans'], df['normalized_score'], test_size=0.3)

# Reset index
train_texts = train_texts.reset_index(drop=True)
temp_texts = temp_texts.reset_index(drop=True)
train_scores = train_scores.reset_index(drop=True)
temp_scores = temp_scores.reset_index(drop=True)

# val data and test data
val_texts, test_texts, val_scores, test_scores = train_test_split(temp_texts, temp_scores, test_size=0.5)

# Reset index
val_texts = val_texts.reset_index(drop=True)
test_texts = test_texts.reset_index(drop=True)
val_scores = val_scores.reset_index(drop=True)
test_scores = test_scores.reset_index(drop=True)


train_dataset = EssayDataset(train_texts, train_scores, tokenizer, max_len=512)
val_dataset = EssayDataset(val_texts, val_scores, tokenizer, max_len=512)
test_dataset = EssayDataset(test_texts, test_scores, tokenizer, max_len=512)

train_loader = DataLoader(train_dataset, batch_size=8)
val_loader = DataLoader(val_dataset, batch_size=8)
test_loader = DataLoader(test_dataset, batch_size=8)

# Training
optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        scores = batch['score'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.squeeze()
        loss = torch.nn.functional.mse_loss(logits, scores)
        total_train_loss += loss.item()
        loss.backward()
        optimizer.step()
    avg_train_loss = total_train_loss / len(train_loader)

    # Validation
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            scores = batch['score'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits.squeeze()
            loss = torch.nn.functional.mse_loss(logits, scores)
            total_val_loss += loss.item()
    avg_val_loss = total_val_loss / len(val_loader)

    print(f'Epoch {epoch + 1}/{num_epochs} | Train Loss: {avg_train_loss} | Val Loss: {avg_val_loss}')

# max_score = df['score'].max()
# Evaluation
def evaluate_model(model, test_loader):
    model.eval()
    predictions, actuals = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            scores = batch['score'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            predictions.extend(outputs.logits.squeeze().cpu().numpy())
            actuals.extend(scores.cpu().numpy())

    predictions = np.array(predictions)
    actuals = np.array(actuals)
    # print("Predictions: ", predictions)
    # print("Actuals: ", actuals)

    predictions_inv_tras = (scaler.inverse_transform(predictions.reshape(-1, 1)).squeeze())
    actuals_inv_tras = (scaler.inverse_transform(actuals.reshape(-1, 1)).squeeze())
    # print("Predictions_inv_tras: ", predictions_inv_tras)
    # print("Actuals_inv_tras: ", actuals_inv_tras)


    mse = mean_squared_error(actuals, predictions)
    mae = mean_absolute_error(actuals, predictions)
    r2 = r2_score(actuals, predictions)
    pearson_corr = pearsonr(actuals, predictions)[0]
    spearman_corr = spearmanr(actuals, predictions)[0]
    qwk = cohen_kappa_score(np.round(actuals_inv_tras), np.round(predictions_inv_tras), weights='quadratic')

    return mse, mae, r2, pearson_corr, spearman_corr, qwk

mse, mae, r2, pearson_corr, spearman_corr, qwk = evaluate_model(model, test_loader)
print(f'MSE: {mse}')
print(f'MAE: {mae}')
print(f'R2: {r2}')
print(f'Pearson Correlation: {pearson_corr}')
print(f'Spearman Correlation: {spearman_corr}')
print(f'QWK: {qwk}')

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at Shushant/NepNewsBERT and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1/10 | Train Loss: 0.060417592472731485 | Val Loss: 0.015281117438072605
Epoch 2/10 | Train Loss: 0.02591313409357099 | Val Loss: 0.008824076852761209
Epoch 3/10 | Train Loss: 0.024651437350257766 | Val Loss: 0.014694868986095701
Epoch 4/10 | Train Loss: 0.024861039411916863 | Val Loss: 0.012780211067625455
Epoch 5/10 | Train Loss: 0.019101001766102854 | Val Loss: 0.013110342269231166
Epoch 6/10 | Train Loss: 0.01851845907367533 | Val Loss: 0.01715009842467095
Epoch 7/10 | Train Loss: 0.0159947386746353 | Val Loss: 0.0074898253411187655
Epoch 8/10 | Train Loss: 0.01606498597902828 | Val Loss: 0.009284363876629089
Epoch 9/10 | Train Loss: 0.01651520710584009 | Val Loss: 0.020202265919319222
Epoch 10/10 | Train Loss: 0.015310552895243745 | Val Loss: 0.007139728654042951
MSE: 0.006879614666104317
MAE: 0.06669460982084274
R2: 0.44499808197270985
Pearson Correlation: 0.6801145715137429
Spearman Correlation: 0.6296982281590052
QWK: 0.6375922792271789


MSE: 0.010290903970599174
MAE: 0.08182667940855026
R2: 0.546128664162972
Pearson Correlation: 0.7812119839124011
Spearman Correlation: 0.7501136354704793
QWK: 0.7233306576410324

### Rajan/NepaliBERT

In [138]:
# Custom Dataset
class EssayDataset(Dataset):
    def __init__(self, essays, scores, tokenizer, max_len):
        self.essays = essays
        self.scores = scores
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.essays)

    def __getitem__(self, item):
        essay = str(self.essays[item])
        score = self.scores[item]

        encoding = self.tokenizer.encode_plus(
            essay,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'score': torch.tensor(score, dtype=torch.float)
        }

# # Model and Tokenizer
# tokenizer = RobertaTokenizer.from_pretrained('amitness/roberta-base-ne')
# model = RobertaForSequenceClassification.from_pretrained('amitness/roberta-base-ne', num_labels=1)
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = model.to(device)


tokenizer = BertTokenizer.from_pretrained('Rajan/NepaliBERT')
model = BertForSequenceClassification.from_pretrained('Rajan/NepaliBERT', num_labels=1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Splitting data
train_texts, temp_texts, train_scores, temp_scores = train_test_split(df['p8_google_trans'], df['normalized_score'], test_size=0.3)

# Reset index
train_texts = train_texts.reset_index(drop=True)
temp_texts = temp_texts.reset_index(drop=True)
train_scores = train_scores.reset_index(drop=True)
temp_scores = temp_scores.reset_index(drop=True)

# val data and test data
val_texts, test_texts, val_scores, test_scores = train_test_split(temp_texts, temp_scores, test_size=0.5)

# Reset index
val_texts = val_texts.reset_index(drop=True)
test_texts = test_texts.reset_index(drop=True)
val_scores = val_scores.reset_index(drop=True)
test_scores = test_scores.reset_index(drop=True)


train_dataset = EssayDataset(train_texts, train_scores, tokenizer, max_len=512)
val_dataset = EssayDataset(val_texts, val_scores, tokenizer, max_len=512)
test_dataset = EssayDataset(test_texts, test_scores, tokenizer, max_len=512)

train_loader = DataLoader(train_dataset, batch_size=8)
val_loader = DataLoader(val_dataset, batch_size=8)
test_loader = DataLoader(test_dataset, batch_size=8)

# Training
optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        scores = batch['score'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.squeeze()
        loss = torch.nn.functional.mse_loss(logits, scores)
        total_train_loss += loss.item()
        loss.backward()
        optimizer.step()
    avg_train_loss = total_train_loss / len(train_loader)

    # Validation
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            scores = batch['score'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits.squeeze()
            loss = torch.nn.functional.mse_loss(logits, scores)
            total_val_loss += loss.item()
    avg_val_loss = total_val_loss / len(val_loader)

    print(f'Epoch {epoch + 1}/{num_epochs} | Train Loss: {avg_train_loss} | Val Loss: {avg_val_loss}')

# max_score = df['score'].max()
# Evaluation
def evaluate_model(model, test_loader):
    model.eval()
    predictions, actuals = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            scores = batch['score'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            predictions.extend(outputs.logits.squeeze().cpu().numpy())
            actuals.extend(scores.cpu().numpy())

    predictions = np.array(predictions)
    actuals = np.array(actuals)
    # print("Predictions: ", predictions)
    # print("Actuals: ", actuals)

    predictions_inv_tras = (scaler.inverse_transform(predictions.reshape(-1, 1)).squeeze())
    actuals_inv_tras = (scaler.inverse_transform(actuals.reshape(-1, 1)).squeeze())
    # print("Predictions_inv_tras: ", predictions_inv_tras)
    # print("Actuals_inv_tras: ", actuals_inv_tras)


    mse = mean_squared_error(actuals, predictions)
    mae = mean_absolute_error(actuals, predictions)
    r2 = r2_score(actuals, predictions)
    pearson_corr = pearsonr(actuals, predictions)[0]
    spearman_corr = spearmanr(actuals, predictions)[0]
    qwk = cohen_kappa_score(np.round(actuals_inv_tras), np.round(predictions_inv_tras), weights='quadratic')

    return mse, mae, r2, pearson_corr, spearman_corr, qwk

mse, mae, r2, pearson_corr, spearman_corr, qwk = evaluate_model(model, test_loader)
print(f'MSE: {mse}')
print(f'MAE: {mae}')
print(f'R2: {r2}')
print(f'Pearson Correlation: {pearson_corr}')
print(f'Spearman Correlation: {spearman_corr}')
print(f'QWK: {qwk}')

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at Rajan/NepaliBERT and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1/10 | Train Loss: 0.04731974147580331 | Val Loss: 0.010882494572017874
Epoch 2/10 | Train Loss: 0.02513242782879388 | Val Loss: 0.014528092063431228
Epoch 3/10 | Train Loss: 0.0269187879312085 | Val Loss: 0.0072238810327170154
Epoch 4/10 | Train Loss: 0.02226131939309539 | Val Loss: 0.007552530084337507
Epoch 5/10 | Train Loss: 0.020758609816766693 | Val Loss: 0.0073187436376299176
Epoch 6/10 | Train Loss: 0.022212757969100494 | Val Loss: 0.007146740232461265
Epoch 7/10 | Train Loss: 0.018227007261884864 | Val Loss: 0.006980135055658009
Epoch 8/10 | Train Loss: 0.018758356545731658 | Val Loss: 0.006790688685474119
Epoch 9/10 | Train Loss: 0.018028645783488173 | Val Loss: 0.0109670472198299
Epoch 10/10 | Train Loss: 0.019374888710444793 | Val Loss: 0.009212774041640972
MSE: 0.012126985006034374
MAE: 0.08553129434585571
R2: 0.2921647975906563
Pearson Correlation: 0.6383809790546604
Spearman Correlation: 0.6284731526969531
QWK: 0.5773952701032248


MSE: 0.009544388391077518
MAE: 0.07698043435811996
R2: 0.6639626400186949
Pearson Correlation: 0.8163915193699683
Spearman Correlation: 0.7688454865656341
QWK: 0.7961585517696863

### dexhrestha/Nepali-DistilBERT

In [139]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("dexhrestha/Nepali-DistilBERT")
max_length = tokenizer.model_max_length
print("Max sequence length:", max_length)

Max sequence length: 128


In [143]:
# seed = 128
# torch.manual_seed(seed)
# torch.cuda.manual_seed(seed)
# torch.cuda.manual_seed_all(seed)

# if torch.cuda.is_available():
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False

# np.random.seed(seed)

# Custom Dataset
class EssayDataset(Dataset):
    def __init__(self, essays, scores, tokenizer, max_len):
        self.essays = essays
        self.scores = scores
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.essays)

    def __getitem__(self, item):
        essay = str(self.essays[item])
        score = self.scores[item]

        encoding = self.tokenizer.encode_plus(
            essay,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'score': torch.tensor(score, dtype=torch.float)
        }


tokenizer = AutoTokenizer.from_pretrained('dexhrestha/Nepali-DistilBERT')
model = AutoModelForSequenceClassification.from_pretrained('dexhrestha/Nepali-DistilBERT', num_labels=1, ignore_mismatched_sizes=True)
# model = DistilBertForSequenceClassification.from_pretrained('dexhrestha/Nepali-DistilBERT', num_labels=1, ignore_mismatched_sizes=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Splitting data
train_texts, temp_texts, train_scores, temp_scores = train_test_split(df['p8_google_trans'], df['normalized_score'], test_size=0.3)

# Reset index
train_texts = train_texts.reset_index(drop=True)
temp_texts = temp_texts.reset_index(drop=True)
train_scores = train_scores.reset_index(drop=True)
temp_scores = temp_scores.reset_index(drop=True)

# val data and test data
val_texts, test_texts, val_scores, test_scores = train_test_split(temp_texts, temp_scores, test_size=0.5)

# Reset index
val_texts = val_texts.reset_index(drop=True)
test_texts = test_texts.reset_index(drop=True)
val_scores = val_scores.reset_index(drop=True)
test_scores = test_scores.reset_index(drop=True)

max_len = 128
train_dataset = EssayDataset(train_texts, train_scores, tokenizer, max_len=max_len)
val_dataset = EssayDataset(val_texts, val_scores, tokenizer, max_len=max_len)
test_dataset = EssayDataset(test_texts, test_scores, tokenizer, max_len=max_len)

train_loader = DataLoader(train_dataset, batch_size=8)
val_loader = DataLoader(val_dataset, batch_size=8)
test_loader = DataLoader(test_dataset, batch_size=8)

# Training
optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        scores = batch['score'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.squeeze()
        loss = torch.nn.functional.mse_loss(logits, scores)
        total_train_loss += loss.item()
        loss.backward()
        optimizer.step()
    avg_train_loss = total_train_loss / len(train_loader)

    # Validation
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            scores = batch['score'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits.squeeze()
            loss = torch.nn.functional.mse_loss(logits, scores)
            total_val_loss += loss.item()
    avg_val_loss = total_val_loss / len(val_loader)

    print(f'Epoch {epoch + 1}/{num_epochs} | Train Loss: {avg_train_loss} | Val Loss: {avg_val_loss}')

# max_score = df['score'].max()
# Evaluation
def evaluate_model(model, test_loader):
    model.eval()
    predictions, actuals = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            scores = batch['score'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            predictions.extend(outputs.logits.squeeze().cpu().numpy())
            actuals.extend(scores.cpu().numpy())

    predictions = np.array(predictions)
    actuals = np.array(actuals)
    # print("Predictions: ", predictions)
    # print("Actuals: ", actuals)

    predictions_inv_tras = (scaler.inverse_transform(predictions.reshape(-1, 1)).squeeze())
    actuals_inv_tras = (scaler.inverse_transform(actuals.reshape(-1, 1)).squeeze())
    # print("Predictions_inv_tras: ", predictions_inv_tras)
    # print("Actuals_inv_tras: ", actuals_inv_tras)


    mse = mean_squared_error(actuals, predictions)
    mae = mean_absolute_error(actuals, predictions)
    r2 = r2_score(actuals, predictions)
    pearson_corr = pearsonr(actuals, predictions)[0]
    spearman_corr = spearmanr(actuals, predictions)[0]
    qwk = cohen_kappa_score(np.round(actuals_inv_tras), np.round(predictions_inv_tras), weights='quadratic')

    return mse, mae, r2, pearson_corr, spearman_corr, qwk

mse, mae, r2, pearson_corr, spearman_corr, qwk = evaluate_model(model, test_loader)
print(f'MSE: {mse}')
print(f'MAE: {mae}')
print(f'R2: {r2}')
print(f'Pearson Correlation: {pearson_corr}')
print(f'Spearman Correlation: {spearman_corr}')
print(f'QWK: {qwk}')

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at dexhrestha/Nepali-DistilBERT and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([1, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([1]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1/10 | Train Loss: 0.04415434697875753 | Val Loss: 0.013165592116170697
Epoch 2/10 | Train Loss: 0.019346812208823394 | Val Loss: 0.012158221831279141
Epoch 3/10 | Train Loss: 0.018179874437919352 | Val Loss: 0.011790610118103879
Epoch 4/10 | Train Loss: 0.01449766620498849 | Val Loss: 0.010956691711076669
Epoch 5/10 | Train Loss: 0.01171670918847667 | Val Loss: 0.010927877156063914
Epoch 6/10 | Train Loss: 0.011031806239770958 | Val Loss: 0.011740914812045438
Epoch 7/10 | Train Loss: 0.009761895505107532 | Val Loss: 0.01651035325734743
Epoch 8/10 | Train Loss: 0.008709818335773889 | Val Loss: 0.023554120512147034
Epoch 9/10 | Train Loss: 0.00931730549018539 | Val Loss: 0.02387765435768025
Epoch 10/10 | Train Loss: 0.012018272824207088 | Val Loss: 0.011480862923365618
MSE: 0.012424922548234463
MAE: 0.08850821852684021
R2: 0.2214371346179328
Pearson Correlation: 0.4837578908150444
Spearman Correlation: 0.4102122431855414
QWK: 0.3765105787901891


MSE: 0.01469359826296568
MAE: 0.09399642050266266
R2: 0.4295974815579148
Pearson Correlation: 0.6651744400495655
Spearman Correlation: 0.5645745537632599
QWK: 0.6028663263918219

In [141]:
# # Set seed for reproducibility
# seed = 128
# torch.manual_seed(seed)
# torch.cuda.manual_seed(seed)
# torch.cuda.manual_seed_all(seed)

# if torch.cuda.is_available():
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False

# np.random.seed(seed)

# # Custom Dataset
# class EssayDataset(Dataset):
#     def __init__(self, essays, scores, tokenizer, max_len):
#         self.essays = essays
#         self.scores = scores
#         self.tokenizer = tokenizer
#         self.max_len = max_len

#     def __len__(self):
#         return len(self.essays)

#     def __getitem__(self, item):
#         essay = str(self.essays[item])
#         score = self.scores[item]

#         encoding = self.tokenizer.encode_plus(
#             essay,
#             add_special_tokens=True,
#             max_length=self.max_len,
#             padding='max_length',
#             return_attention_mask=True,
#             return_tensors='pt',
#             truncation=True
#         )

#         return {
#             'input_ids': encoding['input_ids'].flatten(),
#             'attention_mask': encoding['attention_mask'].flatten(),
#             'score': torch.tensor(score, dtype=torch.float)
#         }

# # Splitting data
# train_texts, temp_texts, train_scores, temp_scores = train_test_split(df['p8_google_trans'], df['normalized_score'], test_size=0.3)
# val_texts, test_texts, val_scores, test_scores = train_test_split(temp_texts, temp_scores, test_size=0.5)

# # Reset index
# train_texts, temp_texts, val_texts, test_texts = map(lambda x: x.reset_index(drop=True), [train_texts, temp_texts, val_texts, test_texts])
# train_scores, temp_scores, val_scores, test_scores = map(lambda x: x.reset_index(drop=True), [train_scores, temp_scores, val_scores, test_scores])

# # Model and Tokenizer
# tokenizer = AutoTokenizer.from_pretrained('dexhrestha/Nepali-DistilBERT')
# model = AutoModelForSequenceClassification.from_pretrained('dexhrestha/Nepali-DistilBERT', num_labels=1, ignore_mismatched_sizes=True)
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = model.to(device)

# max_len = 128
# train_dataset = EssayDataset(train_texts, train_scores, tokenizer, max_len=max_len)
# val_dataset = EssayDataset(val_texts, val_scores, tokenizer, max_len=max_len)
# test_dataset = EssayDataset(test_texts, test_scores, tokenizer, max_len=max_len)

# # Training
# def train_model(trial, model, train_loader, val_loader):
#   try:
#       lr = trial.suggest_loguniform('lr', 1e-6, 1e-3)
#       weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)
#       batch_size = trial.suggest_categorical('batch_size', [4, 8, 16])
#       num_epochs = trial.suggest_int('num_epochs', 1, 3)

#       optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
#       train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
#       val_loader = DataLoader(val_dataset, batch_size=batch_size)

#       for epoch in range(num_epochs):
#           model.train()
#           total_train_loss = 0
#           for batch in train_loader:
#               input_ids = batch['input_ids'].to(device)
#               attention_mask = batch['attention_mask'].to(device)
#               scores = batch['score'].to(device)

#               optimizer.zero_grad()
#               outputs = model(input_ids=input_ids, attention_mask=attention_mask)
#               logits = outputs.logits.squeeze()
#               loss = torch.nn.functional.mse_loss(logits, scores)
#               total_train_loss += loss.item()
#               loss.backward()
#               optimizer.step()

#           avg_train_loss = total_train_loss / len(train_loader)

#           # Validation
#           model.eval()
#           total_val_loss = 0
#           with torch.no_grad():
#               for batch in val_loader:
#                   input_ids = batch['input_ids'].to(device)
#                   attention_mask = batch['attention_mask'].to(device)
#                   scores = batch['score'].to(device)

#                   outputs = model(input_ids=input_ids, attention_mask=attention_mask)
#                   logits = outputs.logits.squeeze()
#                   loss = torch.nn.functional.mse_loss(logits, scores)
#                   total_val_loss += loss.item()

#           avg_val_loss = total_val_loss / len(val_loader)

#           print(f'Epoch {epoch + 1}/{num_epochs} | Train Loss: {avg_train_loss} | Val Loss: {avg_val_loss}')
#           return avg_val_loss
#   except Exception as e:
#           print(f"An error occurred: {e}")
#           return float('inf')  # Return a large value in case of an error

# # Evaluation
# def evaluate_model(model, test_loader):
#     model.eval()
#     predictions, actuals = [], []
#     with torch.no_grad():
#         for batch in test_loader:
#             input_ids = batch['input_ids'].to(device)
#             attention_mask = batch['attention_mask'].to(device)
#             scores = batch['score'].to(device)

#             outputs = model(input_ids=input_ids, attention_mask=attention_mask)
#             predictions.extend(outputs.logits.squeeze().cpu().numpy())
#             actuals.extend(scores.cpu().numpy())

#     predictions = np.array(predictions)
#     actuals = np.array(actuals)
#     # print("Predictions: ", predictions)
#     # print("Actuals: ", actuals)

#     predictions_inv_tras = (scaler.inverse_transform(predictions.reshape(-1, 1)).squeeze())
#     actuals_inv_tras = (scaler.inverse_transform(actuals.reshape(-1, 1)).squeeze())
#     # print("Predictions_inv_tras: ", predictions_inv_tras)
#     # print("Actuals_inv_tras: ", actuals_inv_tras)

#     mse = mean_squared_error(actuals, predictions)
#     mae = mean_absolute_error(actuals, predictions)
#     r2 = r2_score(actuals, predictions)
#     pearson_corr = pearsonr(actuals, predictions)[0]
#     spearman_corr = spearmanr(actuals, predictions)[0]
#     qwk = cohen_kappa_score(np.round(actuals_inv_tras), np.round(predictions_inv_tras), weights='quadratic')

#     return mse, mae, r2, pearson_corr, spearman_corr, qwk

# # Optuna study and optimization
# study = optuna.create_study(direction='minimize')
# study.optimize(lambda trial: train_model(trial, model, train_loader, val_loader), n_trials=50)

# # Get best hyperparameters from the study
# best_lr = study.best_params['lr']
# best_weight_decay = study.best_params['weight_decay']
# best_batch_size = study.best_params['batch_size']
# best_num_epochs = study.best_params['num_epochs']

# print(f'Best LR: {best_lr}')
# print(f'Best Weight Decay: {best_weight_decay}')
# print(f'Best Batch Size: {best_batch_size}')
# print(f'Best Num Epochs: {best_num_epochs}')

# # Use the best hyperparameters to train your final model
# optimizer = AdamW(model.parameters(), lr=best_lr, weight_decay=best_weight_decay)
# train_loader = DataLoader(train_dataset, batch_size=best_batch_size, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=best_batch_size)

# # Evaluate final model
# mse, mae, r2, pearson_corr, spearman_corr, qwk = evaluate_model(model, test_loader)
# print(f'MSE: {mse}')
# print(f'MAE: {mae}')
# print(f'R2: {r2}')
# print(f'Pearson Correlation: {pearson_corr}')
# print(f'Spearman Correlation: {spearman_corr}')
# print(f'QWK: {qwk}')